# 只记录和，会丢掉哪些区别？

我们从两个数出发，不先调用求解器。改变输入、改变记录规则、再增加记录，观察哪些变化能被发现。

依赖 NumPy、Matplotlib；从新内核按顺序运行。图使用英文与数学符号，避免中文字体缺失。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
A = np.array([[1., 1.]])
x0 = np.array([3., 2.])
v = np.array([1., -1.])


## 1. 两个输入，为什么一个输出？

先手算 (3,2)、(4,1) 的和，再算第二组减第一组。差乘上 A 得到什么？

把第二组改成 (4,2)，先预测输出是否相同。

In [ ]:
x1 = np.array([4., 1.])  # Also try [4., 2.].
difference = x1-x0
print("first input:", x0, "output:", A @ x0)
print("second input:", x1, "output:", A @ x1)
print("difference:", difference, "difference output:", A @ difference)
assert np.allclose(A @ difference, A @ x1-A @ x0)


## 2. 找全零输出的输入

方程 x1+x2=0 要求 x2=-x1。先选 t，再预测 (t,-t) 的输出。

不是画几个点就证明找全了：任何解的第二项都必须是第一项的相反数。

In [ ]:
t = 2.5  # Try 0, -3, 100.
zero_input = t*v
print("input:", zero_input, "output:", A @ zero_input)
assert np.allclose(A @ zero_input, 0)


## 3. 沿这个方向移动，输出为什么不变？

给 (3,2) 加 t(1,-1)，再试加 t(1,0)。运行前预测：哪一种变化不可见？

在 t=0 时，两种变化都没发生；请用非零 t 比较。

In [ ]:
t = 1.
print("balanced change:", x0+t*v, "output:", A @ (x0+t*v))
print("one-coordinate change:", x0+t*np.array([1.,0.]),
      "output:", A @ (x0+t*np.array([1.,0.])))
assert np.allclose(A @ (x0+t*v), A @ x0)


## 4. 一个特解加零空间，找回全部解

输出目标为 5，选特解 (5,0)。先算哪个 t 会得到 (3,2)。为什么任选 t 都产生 5？为什么任何和为 5 的输入都能这样写？

In [ ]:
particular = np.array([5., 0.])
t = -2.
candidate = particular+t*v
print("candidate:", candidate, "output:", A @ candidate)
assert np.allclose(A @ candidate, [5])
assert np.allclose(A @ (candidate-particular), 0)


## 5. 两条平行线，各自装什么？

蓝线装零输出的输入，橙线装输出为 5 的输入。图中的点都是二维输入，不是输出。

为什么蓝线经过原点，橙线不经过？橙线上的箭头为什么与蓝线同方向？

In [ ]:
grid = np.linspace(-3, 7, 201)
fig, ax = plt.subplots(figsize=(7, 6), dpi=160)
ax.plot(grid, -grid, color="#2878b5", lw=2.5, label=r"$Ax=0$: null space")
ax.plot(grid, 5-grid, color="#d97922", lw=2.5, label=r"$Ax=5$: all solutions")
for point, label, offset in [
    ((0,0), "Origin", (10,10)), ((5,0), "(5,0)", (10,10)),
    ((3,2), "(3,2)", (-55,-15)), ((4,1), "(4,1)", (12,7))
]:
    ax.scatter(*point, s=40, color="#444444", zorder=4)
    ax.annotate(label, point, xytext=offset, textcoords="offset points", fontsize=10)
ax.annotate("", xy=(4,1), xytext=(3,2),
            arrowprops={"arrowstyle":"->", "color":"#b33f58", "lw":2.5})
ax.axhline(0, color="0.7", lw=0.8)
ax.axvline(0, color="0.7", lw=0.8)
ax.set(xlabel=r"Input $x_1$", ylabel=r"Input $x_2$",
       xlim=(-3,7), ylim=(-3,7), aspect="equal")
ax.grid(alpha=0.2)
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()


## 6. 改规则，重新找不可见的变化

现在记录 x1+2x2。原来的 (1,-1) 还被映成零吗？

先解 z1+2z2=0，选出新的方向，再运行核对。不要只把旧方向沿用过去。

In [ ]:
weighted = np.array([[1., 2.]])
new_direction = np.array([2., -1.])
print("old direction output:", weighted @ v)
print("new direction output:", weighted @ new_direction)
assert not np.allclose(weighted @ v, 0)
assert np.allclose(weighted @ new_direction, 0)


## 7. 再记录差，原来的区别还不可见吗？

同时记录和与差。先算 (3,2)、(4,1) 的两项输出，再算原来不可见的变化 (1,-1) 的输出。

若 z1+z2=0 且 z1-z2=0，z1、z2 只能取什么值？这次能唯一恢复输入吗？

In [ ]:
S = np.array([[1.,1.], [1.,-1.]])
for point in [np.array([3.,2.]), np.array([4.,1.])]:
    print("input:", point, "sum and difference:", S @ point)
print("change output:", S @ v)
p, q = S @ x0
recovered = np.array([(p+q)/2, (p-q)/2])
assert np.allclose(recovered, x0)
assert np.allclose(S @ v, [0,2])


## 自己重建

为什么先找零输出，就能理解相同输出的不同输入？解释 A(x-x0)=0。

把计算规则换成 2x1+3x2，不运行求解器，写出它的全部零输出输入，再写出输出为 6 的全部输入。

下一章比较新增测量何时有效，何时只是把同一份记录写两遍。